# 03 -- Optionality and Cardinality

orthograph distinguishes **three levels of optionality** when validating graph data:

1. **Property optionality** -- individual fields on a node or relationship can be required or optional.
2. **Entity-level optionality** -- an entire node or relationship *type* can be marked as required or optional in the model.
3. **Cardinality** -- constraints on how many relationships of a given type each node may have.

This notebook covers each level in isolation, then shows how they combine in practice.

In [1]:
from typing import Optional

from orthograph import (
    Cardinality,
    CardinalitySpec,
    GraphDataModel,
    GraphValidationError,
    GraphValidator,
    NodeModel,
    RelationshipModel,
)

## Level 1: Property Optionality

Properties on `NodeModel` and `RelationshipModel` follow standard Pydantic rules:

- A field declared as `name: str` is **required** -- validation fails if it is missing.
- A field declared as `email: Optional[str] = None` is **optional** -- it can be absent or `None`.

This is the most granular level of optionality.

In [2]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"

    name: str                        # required
    age: int                         # required
    email: Optional[str] = None      # optional

# Quick model to test node validation in isolation
prop_model = GraphDataModel(name="PropTest", node_types=[Person], relationship_types=[])
v = GraphValidator(prop_model)

# Valid: all required fields present, optional omitted
r = v.validate_nodes([{"__label__": "Person", "name": "Alice", "age": 32}])
print("All required present, optional omitted:", r.is_valid)

# Valid: optional field explicitly set to None
r = v.validate_nodes([{"__label__": "Person", "name": "Alice", "age": 32, "email": None}])
print("Optional field set to None:            ", r.is_valid)

# Invalid: required field 'age' missing
r = v.validate_nodes([{"__label__": "Person", "name": "Alice"}])
print("Required field missing:                ", r.is_valid)
for err in r.errors:
    print(f"  -> {err.message}")

All required present, optional omitted: True
Optional field set to None:             True
Required field missing:                 False
  -> Validation error: Field required (field: age)


## Level 2: Entity-Level Optionality

Every `NodeModel` and `RelationshipModel` has a class variable `__optional__`
(default `True`). This controls whether the *type itself* must have at least one
instance present in the data.

- `__optional__ = True` (the default) -- the model defines what **can** exist.
  It is fine if the data contains zero instances of this type.
- `__optional__ = False` -- the model **requires** at least one instance of this
  type. Validation fails if the data is missing it entirely.

This is useful when you want to guarantee that certain entity types always appear,
for example to enforce that every valid graph contains at least one Person.

In [3]:
class RequiredMovie(NodeModel):
    __label__ = "RequiredMovie"
    __uid_field__ = "title"
    __optional__ = False  # must have at least one instance

    title: str
    year: int


class OptionalCity(NodeModel):
    __label__ = "OptionalCity"
    __uid_field__ = "name"
    __optional__ = True  # this is the default, shown explicitly

    name: str
    country: str


entity_model = GraphDataModel(
    name="EntityTest",
    node_types=[RequiredMovie, OptionalCity],
    relationship_types=[],
)
v2 = GraphValidator(entity_model)

# Empty data: RequiredMovie is missing
r = v2.validate(nodes=[], relationships=[])
print("Empty data -- is_valid:", r.is_valid)
for err in r.errors:
    print(f"  [{err.code}] {err.message}")

print()

# Provide only RequiredMovie -- OptionalCity can be absent
r = v2.validate(
    nodes=[{"__label__": "RequiredMovie", "title": "Inception", "year": 2010}],
    relationships=[],
)
print("Only RequiredMovie present -- is_valid:", r.is_valid)

Empty data -- is_valid: False
  [MISSING_REQUIRED_TYPE] Required node type 'RequiredMovie' has no instances in data

Only RequiredMovie present -- is_valid: True


## Level 3: Cardinality

Cardinality specifies how many relationships of a given type a node can participate in.
It is expressed as a `CardinalitySpec(min, max)` where `max=None` means unbounded.

orthograph provides four named constants on the `Cardinality` class:

| Constant | min | max | Meaning |
|---|---|---|---|
| `ZERO_OR_ONE` | 0 | 1 | At most one |
| `ONE` | 1 | 1 | Exactly one |
| `ZERO_OR_MORE` | 0 | None | Any number (default) |
| `ONE_OR_MORE` | 1 | None | At least one |

You can also create custom specs with `CardinalitySpec(min=..., max=...)`.

Cardinality is set per direction:
- `__source_cardinality__` constrains how many outgoing rels of this type each source node has.
- `__target_cardinality__` constrains how many incoming rels of this type each target node has.

In [4]:
# Named constants
for name in ["ZERO_OR_ONE", "ONE", "ZERO_OR_MORE", "ONE_OR_MORE"]:
    spec = getattr(Cardinality, name)
    max_str = "N" if spec.max is None else str(spec.max)
    print(f"  Cardinality.{name:15s}  min={spec.min}  max={max_str}")

print()

# Custom cardinality
custom = CardinalitySpec(min=2, max=5)
print(f"Custom spec: {custom}")
print(f"  contains(1) = {custom.contains(1)}")
print(f"  contains(3) = {custom.contains(3)}")
print(f"  contains(6) = {custom.contains(6)}")

  Cardinality.ZERO_OR_ONE      min=0  max=1
  Cardinality.ONE              min=1  max=1
  Cardinality.ZERO_OR_MORE     min=0  max=N
  Cardinality.ONE_OR_MORE      min=1  max=N

Custom spec: min=2 max=5
  contains(1) = False
  contains(3) = True
  contains(6) = False


## Cardinality in Practice

Let's define a model where each Person must have exactly one LIVES_IN
relationship (`__source_cardinality__ = Cardinality.ONE`), meaning every Person
must live in exactly one City. We then validate data that violates this constraint
in both directions: too few and too many.

In [5]:
class CPerson(NodeModel):
    __label__ = "CPerson"
    __uid_field__ = "name"
    name: str

class CCity(NodeModel):
    __label__ = "CCity"
    __uid_field__ = "name"
    name: str

class CLivesIn(RelationshipModel):
    __label__ = "C_LIVES_IN"
    __source_type__ = CPerson
    __target_type__ = CCity
    __source_cardinality__ = Cardinality.ONE  # each person -> exactly 1 city
    __target_cardinality__ = Cardinality.ZERO_OR_MORE

card_model = GraphDataModel(
    name="CardinalityDemo",
    node_types=[CPerson, CCity],
    relationship_types=[CLivesIn],
)
v3 = GraphValidator(card_model)

base_nodes = [
    {"__label__": "CPerson", "name": "Alice"},
    {"__label__": "CCity", "name": "London"},
    {"__label__": "CCity", "name": "Paris"},
]

# --- Too few: Alice has 0 LIVES_IN (needs exactly 1) ---
r = v3.validate(nodes=base_nodes, relationships=[])
print("Too few (0 LIVES_IN):")
print(f"  is_valid: {r.is_valid}")
for err in r.errors:
    print(f"  [{err.code}] {err.message}")

print()

# --- Too many: Alice has 2 LIVES_IN (needs exactly 1) ---
r = v3.validate(
    nodes=base_nodes,
    relationships=[
        {"__label__": "C_LIVES_IN", "__source_uid__": "Alice", "__target_uid__": "London"},
        {"__label__": "C_LIVES_IN", "__source_uid__": "Alice", "__target_uid__": "Paris"},
    ],
)
print("Too many (2 LIVES_IN):")
print(f"  is_valid: {r.is_valid}")
for err in r.errors:
    print(f"  [{err.code}] {err.message}")

print()

# --- Just right: exactly 1 ---
r = v3.validate(
    nodes=base_nodes,
    relationships=[
        {"__label__": "C_LIVES_IN", "__source_uid__": "Alice", "__target_uid__": "London"},
    ],
)
print("Exactly 1 LIVES_IN:")
print(f"  is_valid: {r.is_valid}")

Too few (0 LIVES_IN):
  is_valid: False
  [CARDINALITY_VIOLATION] Node 'Alice' (CPerson) has 0 outgoing C_LIVES_IN relationships, expected 1..1

Too many (2 LIVES_IN):
  is_valid: False
  [CARDINALITY_VIOLATION] Node 'Alice' (CPerson) has 2 outgoing C_LIVES_IN relationships, expected 1..1

Exactly 1 LIVES_IN:
  is_valid: True


## Understanding ZERO_OR_MORE: Cardinality vs. Existence

A common question is: *"Why does `ZERO_OR_MORE` exist? If a node has zero
relationships of that type, doesn't that mean the relationship doesn't exist?"*

The answer lies in distinguishing two **orthogonal** concepts:

| Concept | What it controls | Mechanism |
|---|---|---|
| **Entity-level optionality** | Whether the relationship *type* must appear at all in the data | `__optional__ = True/False` |
| **Cardinality** | How many instances each *individual node* may have | `CardinalitySpec(min, max)` |

`ZERO_OR_MORE` (0..\*) means: *"this relationship type is defined in the schema,
but individual nodes are not required to participate in it."* A count of zero is a
valid state -- the node simply has no such relationship. This is **not** the same as
saying the relationship type doesn't exist.

`ONE_OR_MORE` (1..\*) means: *"every node of this type must have at least one
instance of this relationship."* Zero would be a violation.

This is standard UML/ER notation -- `0..*` and `1..*` are both well-defined
multiplicities with distinct semantics.

The following example demonstrates the difference concretely.

In [6]:
# --- Side-by-side: ZERO_OR_MORE vs ONE_OR_MORE ---

class Employee(NodeModel):
    __label__ = "Employee"
    __uid_field__ = "name"
    name: str

class Project(NodeModel):
    __label__ = "Project"
    __uid_field__ = "name"
    name: str

# Relaxed: employees MAY work on projects (but don't have to)
class WorksOnRelaxed(RelationshipModel):
    __label__ = "WORKS_ON_R"
    __source_type__ = Employee
    __target_type__ = Project
    __source_cardinality__ = Cardinality.ZERO_OR_MORE  # 0..* -- optional
    __target_cardinality__ = Cardinality.ZERO_OR_MORE

# Strict: every employee MUST work on at least one project
class WorksOnStrict(RelationshipModel):
    __label__ = "WORKS_ON_S"
    __source_type__ = Employee
    __target_type__ = Project
    __source_cardinality__ = Cardinality.ONE_OR_MORE   # 1..* -- mandatory
    __target_cardinality__ = Cardinality.ZERO_OR_MORE

relaxed_model = GraphDataModel(
    name="Relaxed", node_types=[Employee, Project],
    relationship_types=[WorksOnRelaxed],
)
strict_model = GraphDataModel(
    name="Strict", node_types=[Employee, Project],
    relationship_types=[WorksOnStrict],
)

nodes = [
    {"__label__": "Employee", "name": "Alice"},
    {"__label__": "Employee", "name": "Bob"},
    {"__label__": "Project", "name": "Alpha"},
]

# Only Alice works on Alpha; Bob has zero relationships
rels = [
    {"__label__": "WORKS_ON_R", "__source_uid__": "Alice", "__target_uid__": "Alpha"},
]
rels_strict = [
    {"__label__": "WORKS_ON_S", "__source_uid__": "Alice", "__target_uid__": "Alpha"},
]

# --- Relaxed model: Bob with 0 relationships is fine ---
r = GraphValidator(relaxed_model).validate(nodes, rels)
print("ZERO_OR_MORE (relaxed):")
print(f"  is_valid: {r.is_valid}")
print(f"  Bob has 0 WORKS_ON -- accepted (participation is optional)")

print()

# --- Strict model: Bob with 0 relationships is a violation ---
r = GraphValidator(strict_model).validate(nodes, rels_strict)
print("ONE_OR_MORE (strict):")
print(f"  is_valid: {r.is_valid}")
for err in r.errors:
    print(f"  [{err.code}] {err.message}")
print(f"  Bob has 0 WORKS_ON -- rejected (participation is mandatory)")

ZERO_OR_MORE (relaxed):
  is_valid: True
  Bob has 0 WORKS_ON -- accepted (participation is optional)

ONE_OR_MORE (strict):
  is_valid: False
  [CARDINALITY_VIOLATION] Node 'Bob' (Employee) has 0 outgoing WORKS_ON_S relationships, expected 1..N
  Bob has 0 WORKS_ON -- rejected (participation is mandatory)


### When to use each

**Use `ZERO_OR_MORE`** (the default) when:
- Validating **partial query results** where not all relationships are returned
- The relationship is genuinely optional (not every Person has a REVIEWED relationship)
- You want the schema to document what *can* exist without enforcing participation

**Use `ONE_OR_MORE`** when:
- Every node of this type **must** participate (every Employee must WORK_AT somewhere)
- You are validating **canonical/complete data**, not query fragments
- The business rule requires mandatory participation with no upper bound

Both are valid, well-defined cardinalities. The choice depends on the strictness
of your domain rules and whether you expect complete or partial data.

## Combining Optionality Levels

In practice, these three levels work together. Consider a common scenario:
you receive partial data from a query and want a **relaxed** model that
accepts incomplete results.

- Some node types are **optional** (`__optional__ = True`) -- they might not appear.
- Some properties are **optional** (`Optional[T] = None`) -- they might be null.
- Cardinality is **relaxed** (`ZERO_OR_MORE`) -- we do not enforce relationship counts.

Contrast this with a **strict** model for canonical data, where types are required,
properties are mandatory, and cardinality is enforced.

In [7]:
# -- A relaxed "query result" model --

class QRPerson(NodeModel):
    __label__ = "QRPerson"
    __uid_field__ = "name"
    __optional__ = True  # might not appear in partial results

    name: str
    age: Optional[int] = None       # age might not be projected
    email: Optional[str] = None


class QRMovie(NodeModel):
    __label__ = "QRMovie"
    __uid_field__ = "title"
    __optional__ = True

    title: str
    year: Optional[int] = None      # might not be projected
    rating: Optional[float] = None


class QRCity(NodeModel):
    __label__ = "QRCity"
    __uid_field__ = "name"
    __optional__ = True

    name: str
    country: Optional[str] = None


class QRActedIn(RelationshipModel):
    __label__ = "QR_ACTED_IN"
    __source_type__ = QRPerson
    __target_type__ = QRMovie
    __source_cardinality__ = Cardinality.ZERO_OR_MORE  # relaxed
    __target_cardinality__ = Cardinality.ZERO_OR_MORE

    role: Optional[str] = None  # role might not be projected


class QRLivesIn(RelationshipModel):
    __label__ = "QR_LIVES_IN"
    __source_type__ = QRPerson
    __target_type__ = QRCity
    __source_cardinality__ = Cardinality.ZERO_OR_MORE  # relaxed (not ONE)
    __target_cardinality__ = Cardinality.ZERO_OR_MORE


query_model = GraphDataModel(
    name="QueryResult",
    node_types=[QRPerson, QRMovie, QRCity],
    relationship_types=[QRActedIn, QRLivesIn],
)
v_query = GraphValidator(query_model)

# Partial data: only persons and one relationship, no movies/cities present
partial_nodes = [
    {"__label__": "QRPerson", "name": "Alice"},           # age missing (optional)
    {"__label__": "QRPerson", "name": "Bob", "age": 28},
    {"__label__": "QRMovie", "title": "Inception"},        # year missing (optional)
]

partial_rels = [
    {"__label__": "QR_ACTED_IN", "__source_uid__": "Alice", "__target_uid__": "Inception"},
]

r = v_query.validate(partial_nodes, partial_rels)
print("Partial query result against relaxed model:")
print(f"  is_valid: {r.is_valid}")
print(f"  errors:   {len(r.errors)}")
print(f"  warnings: {len(r.warnings)}")
print()
print("All three optionality levels cooperate: optional types can be absent,")
print("optional properties can be null, and relaxed cardinality allows any count.")

Partial query result against relaxed model:
  is_valid: True
  errors:   0
  warnings: 0

All three optionality levels cooperate: optional types can be absent,
optional properties can be null, and relaxed cardinality allows any count.
